# Discrete models: masked & uniform diffusion

So far the generative models considered in the tutorials interpolated between *continuous* variables and sampled an O/SDE. In this tutorial we demonstrate how to implement models with **discrete** modalities interpolated in **discrete** space. 

As for the previous tutorials, *we strongly recommend you read the [introduction](https://instadeepai.github.io/stix/introduction.html) page* before you dive into this tutorial.

In this notebook, we will consider two popular discrete generative models, namely **masked diffusion** and **uniform diffusion**, and then a **joint** multi-modal model including both continuous and discrete modalities.

## Structure of the notebook

1. Imports
2. A toy discrete dataset
3. **Masked diffusion**
4. **Uniform diffusion**
5. **Joint multi-modal model**

## 0. Installation

We recommend running this notebook in a **fresh virtual environment** (see [1.training_and_sampling.ipynb](./1.training_and_sampling.ipynb) for the full setup). Then install `stix`:

In [ ]:
%pip install stix-ml scikit-learn matplotlib seaborn

## 1. Imports

In [ ]:
import logging
from functools import partial

import grain
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import optax
from flax import nnx
from matplotlib.colors import BoundaryNorm

from stix.core.embedder import IdentityEmbedder, OneHotDiscreteEmbedder
from stix.core.gen_model.factory import (
    PosteriorMixtureGenerativeModel,
    VelocityAndPosteriorGenerativeModel,
)

# The continuous one-sided interpolant, plus the discrete mixture path.
from stix.core.interpolant import (
    FlowMatchingOneSidedInterpolant,
    MaskDiscreteInterpolant,
    UniformDiscreteInterpolant,
)
from stix.core.modality import ModalityRegistry

# DiT building blocks + the thin EncoderBackboneDecoderNetwork wrapper that orchestrates them.
from stix.nn import (
    DiTBackbone,
    DiTDecoder,
    DiTEncoder,
    EncoderBackboneDecoderNetwork,
    NetworkDimsConfig,
    SumContextEncoder,
    TimeNoiseContextEncoder,
)

# The fixed-step solver: the only one that can step a CTMC (TransitionRates).
from stix.sampling.solver_manual import ManualSolver, ManualSolverConfig
from stix.sampling.utils import Direction
from stix.training.loss_pipeline import LossPipeline
from stix.training.training_io_handler import LoggingCategory, TrainingIOHandler
from stix.training.training_loggers import log_metrics_to_line
from stix.training.training_loop import TrainingLoop, TrainingLoopConfig
from stix.typing import Batch, RawSourceTargetPair

In [ ]:
stix_logger = logging.getLogger("stix")
stix_logger.setLevel(logging.INFO)

key = jr.PRNGKey(0)

# Toy problem sizes: sequences of decimal digits (see the dataset below).
NUM_CATEGORIES = 10  # K data categories: the digits 0..9
SEQ_LEN = 6  # positions per sequence: two digits each for a, b and c

## 2. A toy discrete dataset: modular multiplication

We will use a toy dataset with purely discrete modalities, which we build as follows. We draw two two-digit numbers $a, b \in \{0, \dots, 99\}$, form their product modulo 100,

$$c = (a \times b) \bmod 100 \in \{0, \dots, 99\},$$

and lay out the six decimal digits of $a$, $b$, $c$ as one sequence:

$$[\underbrace{a_{0},\, a_{1}}_{a},\ \underbrace{b_{0},\, b_{1}}_{b},\ \underbrace{c_{0},\, c_{1}}_{c}].$$

Each position is a digit in $\{0, \dots, 9\}$ and the sequence has $\text{SEQ\_LEN} = 6$ elements.

As in the other tutorials we wrap the sampler in a `grain` dataset. The raw target is an **integer index array**.

In [ ]:
def digits_of(value: jax.Array) -> jax.Array:
    """Two decimal digits (tens, units) of a value in {0, ..., 99}."""
    return jnp.stack([value // 10, value % 10], dtype=jnp.int32)


def sample_multiplication_sequence(key: jax.Array) -> jax.Array:
    """Draw a, b in {0..99}; encode [a, b, (a*b) % 100] as six decimal digits."""
    a_key, b_key = jr.split(key)
    a = jr.randint(a_key, (), 0, 100)
    b = jr.randint(b_key, (), 0, 100)
    c = (a * b) % 100
    digits = jnp.concatenate([digits_of(a), digits_of(b), digits_of(c)])
    return digits.astype(jnp.int32)  # (SEQ_LEN,) = (6,)


@jax.jit
def sample_digits(batch_indices: jax.Array, key: jax.Array) -> Batch:
    """Build a Batch of integer-index digit sequences (source=None: mixture path)."""
    keys = jax.vmap(partial(jax.random.fold_in, key))(batch_indices)
    digits = jax.vmap(sample_multiplication_sequence)(keys)  # (B, SEQ_LEN) int32
    return Batch(
        raw_batch={"tokens": RawSourceTargetPair(target=digits, source=None)},
        is_discrete={"tokens": True},
    )


batch_size = 256


def tokens_dataset(seed: int):
    """An infinite, shuffled, batched stream of multiplication-digit sequences."""
    return (
        grain.MapDataset.range(int(1e9))
        .seed(seed)
        .shuffle()
        .repeat()
        .batch(batch_size, drop_remainder=True)
        .map(partial(sample_digits, key=jr.key(seed)))
        .to_iter_dataset()
    )


train_iter = iter(tokens_dataset(seed=0))
validation_iter = iter(tokens_dataset(seed=42))

In [ ]:
# A few draws, decoded back into the multiplication they encode.
for row in next(validation_iter).raw_batch["tokens"].target[:5]:
    digits = [int(d) for d in row]
    a = digits[0] * 10 + digits[1]
    b = digits[2] * 10 + digits[3]
    c = digits[4] * 10 + digits[5]
    print(f"{digits}  ->  {a:2d} x {b:2d} = {c:2d}  (mod 100)")

## 3. Masked diffusion

We start with a masked diffusion model. Discrete modalities are interpolated in discrete space with a [`MaskDiscreteInterpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.MaskDiscreteInterpolant), a one-sided interpolant in the Discrete Flow Matching (DFM) family (see the [introduction](https://instadeepai.github.io/stix/introduction.html)).

The state space is $\Sigma = \{0,\dots,K-1,m\}$: $K$ data categories plus a dedicated mask symbol $m$ at index $K$ (`num_states = K + 1`). Raw tokens are integer indices; a [`OneHotDiscreteEmbedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.OneHotDiscreteEmbedder) maps them to **one-hot** vectors of width $K+1$, and interpolation (and the CTMC) takes place in that embedded space. Unlike [tutorials 1–5](./1.training_and_sampling.ipynb), these one-hots are treated as *discrete* states, not as continuous vectors.

Given a target token $z_{\mathrm{tgt}}$, the interpolant path is

$$p_t(z \mid z_{\mathrm{tgt}}) = \kappa_t\,\delta_{z_{\mathrm{tgt}}}(z) + (1-\kappa_t)\,\delta_{m}(z),$$

where $\delta_{s}$ is the point mass on state $s$ (the distribution putting probability 1 on $s$ and 0 elsewhere), and $\kappa_t \in [0, 1]$ is the schedule interpolating between them. The schedule is chosen so that $\dot\kappa_t\geq 0$, which corresponds to a progressive unmasking. Here we will take $\kappa_t = t$. At $t=0$ every position is the mask; at $t=1$ it is the target token.

The conditional marginals above can be written in the standard DFM mixture format, namely

$$
p_t(z \mid z_{\mathrm{src}}, z_{\mathrm{tgt}})
   = \sum_{j=1}^{M} \kappa^j_t\, w^j(z \mid z_{\mathrm{src}}, z_{\mathrm{tgt}}),
$$

by taking $M=2$, $\kappa^1_t = \kappa_t$, $w^1(z\mid z_{\mathrm{tgt}}) = \delta_{z_{\mathrm{tgt}}}(z)$, $\kappa^2_t = 1 - \kappa_t$ and $w^2(z\mid z_{\mathrm{tgt}}) = \delta_{m}(z)$.

Recall from the [introduction](https://instadeepai.github.io/stix/introduction.html) that sampling is done through a continuous-time Markov chain (CTMC) whose *probability velocity* $u_t(z\mid y)$ is chosen so as to reproduce the marginals. More precisely, at sampling time the probability velocity allows one to sample $z_{t+\delta t}$ given $z_t$ according to

$$
p_{t+\delta t \mid t}(z \mid y)
= \delta_{y}(z) + \delta t\, u_t(z \mid y) + o(\delta t).
$$

Here $y$ and $z$ correspond respectively to the current state $z_t$ and the future state $z_{t+\delta t}$.

Following [Gat et al., NeurIPS 2024](https://proceedings.neurips.cc/paper_files/paper/2024/hash/f0d629a734b56a642701bba7bc8bb3ed-Abstract-Conference.html), for the generic DFM conditional marginals, such a probability velocity can be written

$$
   u_t(z \mid y, (z_{\mathrm{src}}, z_{\mathrm{tgt}}))
   = \sum_{j=1}^{M} a^j_t\, w^j(z \mid  (z_{\mathrm{src}}, z_{\mathrm{tgt}}))
   + b_t\,\delta_y(z),
$$
with $a^j_t = \dot\kappa^j_t - \kappa^j_t\,\dot\kappa^\ell_t / \kappa^\ell_t$, $b_t = \dot\kappa^\ell_t / \kappa^\ell_t$, and  $\ell = \arg\min_j \dot\kappa^j_t / \kappa^j_t$.

For the two-point schedule above, we have 

$$
\frac{\dot\kappa^1_t}{\kappa^1_t} = \frac{\dot\kappa_t}{\kappa_t},\quad \frac{\dot\kappa^2_t}{\kappa^2_t} = \frac{-\dot\kappa_t}{1-\kappa_t}\,,
$$
and since $\dot\kappa_t\geq 0$ and $0\leq \kappa_t \leq 1$, we get $\ell = 2$. Injecting the expressions of the $w^j$ distributions above, this yields

$$
u_t(z\mid y, z_{\mathrm{tgt}}) = \frac{\dot\kappa_t}{1-\kappa_t}\bigl(\delta_{z_{\mathrm{tgt}}}(z) - \delta_{y}(z)\bigr).
$$

Integrating against the posterior distribution $p_{\mathrm{tgt}\mid t}(z_{\mathrm{tgt}} \mid y)$ (recall that $y$ represents the value of $z_t$ in these expressions), we obtain the *forward* probability velocity

$$\hat{u}_t(z\mid y) = \frac{\dot\kappa_t}{1-\kappa_t}\bigl(p_{\mathrm{tgt}\mid t}(z \mid y) - \delta_{y}(z)\bigr),$$

i.e. $\hat{u}_t = \frac{1}{1-t}\bigl(p_{\mathrm{tgt}\mid t} - \delta_{z_t}\bigr)$ for $\kappa_t = t$. The velocity is nonzero only while $z_t = m$: once a token is revealed it stays put.

The **backward** probability velocity one comes out of the very same derivation run with $\dot\kappa \to -\dot\kappa$: the two ratios become 

$$
\frac{\dot\kappa^1_t}{\kappa^1_t} = -\frac{\dot\kappa_t}{\kappa_t}, \quad \frac{\dot\kappa^2_t}{\kappa^2_t} =\frac{\dot\kappa_t}{(1-\kappa_t)}\,,
$$

so the argmin moves to $\ell = 1$ — that switch is all there is to reversing the chain — giving $b_t = -\dot\kappa_t/\kappa_t$, $a^1_t = 0$ and $a^2_t = \dot\kappa_t/\kappa_t$, hence

$$\check{u}_t(z\mid y) = \frac{\dot\kappa_t}{\kappa_t}\bigl(\delta_{m}(z) - \delta_{y}(z)\bigr),$$

i.e. $\check{u}_t = \frac{1}{t}\bigl(\delta_m - \delta_{z_t}\bigr)$ for $\kappa_t = t$. It mirrors the forward statement: a revealed token is sent back to the mask at rate $\dot\kappa_t/\kappa_t$, and the velocity vanishes once a position is on $m$. Note that $a^1_t = 0$, so — unlike $\hat u_t$ — the backward velocity does not involve the learned posterior at all: re-masking needs no network.

We train a network to predict the denoising posterior $p_{\mathrm{tgt}\mid t}$ over the $K$ *data* categories, then convert it to [`TransitionRates`](https://instadeepai.github.io/stix/api_reference/core/generator.html#stix.core.generator.TransitionRates) via the interpolant's `rates_from_target_posterior`. That method carries both directions: it returns $\hat u_t$ with `backward=False`, and $\check u_t$ with `backward=True`. Both are valid transition rates, and they land in the `forward_rates` and `backward_rates` fields of `TransitionRates`, which the solver mixes as $\bar u_t = (1 + \lambda_t)\,\hat u_t + \lambda_t\,\check u_t$. Section 3.6 puts that mix to work.

### 3.1. Modality registry

As in [tutorial 1](./1.training_and_sampling.ipynb), `ModalityRegistry.from_batch` builds the skeleton from a batch (shape and `is_discrete`). We then fill:

- **interpolant** — a `MaskDiscreteInterpolant` with $K=$ `NUM_CATEGORIES` and $\kappa_t = t$.
- **embedder** — a [`OneHotDiscreteEmbedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.OneHotDiscreteEmbedder) with `num_states = interpolant.num_states` ($K+1$), so the mask symbol has its own one-hot slot. Raw data stay integer indices; the embedder one-hots them.
- **`num_categories`** — $K$, required by the posterior factory model below.


In [ ]:
batch = next(train_iter)

mask_registry = ModalityRegistry.from_batch(batch)
mask_registry.set(
    "interpolant",
    MaskDiscreteInterpolant(num_categories=NUM_CATEGORIES, kappa_fn=lambda t: t),
)
# The embedder is built per modality (`is_factory=True`) so it can read the
# one-hot width off that modality's own interpolant: K + 1 here.
mask_registry.set(
    field="embedder",
    value=lambda modality: OneHotDiscreteEmbedder(
        dm_shape=modality.shape,
        num_states=modality.interpolant.num_states,
    ),
    is_factory=True,
)
# Raw tokens are integer indices, so `from_batch` cannot infer K: state it.
mask_registry.set("num_categories", NUM_CATEGORIES)

### 3.2. Network

Any [`Network`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.Network) can be plugged in. Here the data are a sequence of tokens, so we use the shipped [`EncoderBackboneDecoderNetwork`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.EncoderBackboneDecoderNetwork) with DiT encoder / backbone / decoder (the same stack as [tutorial 5](./5.coupling.ipynb)).

Discrete heads emit $K$ logits (`modality.num_categories`), not $K+1$: the mask symbol is excluded from the posterior. The decoder also has to know whether a modality *has* a sequence axis: the six digit positions form a length-`SEQ_LEN` sequence and keep theirs, whereas a single-token modality (section 5) wants its length-1 axis squeezed away. That is readable off the embedded shape — `(SEQ_LEN, num_states)` has a sequence axis, `(num_states,)` does not — so `make_dit_network` derives it per modality rather than taking it as an argument.

In [ ]:
key, model_key = jr.split(key)
rngs = nnx.Rngs(model_key)

# Some dimensions shared across the encoder, backbone, decoder and context encoder.
# Declared once here, then injected into each component of the network.
network_dims = NetworkDimsConfig(
    embedding_dim=128,
    context_dim=64,
    ffn_hidden_dim=256,
)


def make_dit_network(
    registry,
    rngs,
    *,
    modality_num_tokens,
    dims=None,
    gamma_fn=lambda t: t,
):
    """DiT encoder / backbone / decoder stack for a registry of modalities.

    `gamma_fn` is the continuous noise schedule the time encoder embeds as
    log(gamma_t), so the network can tell t from 1 - t. Discrete interpolants have
    no gamma, so the default is a placeholder standing in for "no noise level";
    pass the real schedule whenever a continuous modality is present.
    """
    dims = network_dims if dims is None else dims
    encoders = registry.map(
        lambda modality, n_tokens: DiTEncoder(
            dims,
            input_dim=modality.embedder.embedding_shape[-1],  # = num_states
            num_tokens=n_tokens,
            rngs=rngs,
        ),
        modality_num_tokens,
    )
    decoders = registry.map(
        lambda modality: DiTDecoder(
            dims,
            output_dim=(
                modality.num_categories
                if modality.is_discrete
                else modality.embedder.embedding_shape[-1]  # continuous: the data dim
            ),
            # A modality with no sequence axis (embedded shape `(num_states,)`)
            # gets its length-1 token axis squeezed back out by the decoder.
            squeeze_sequence=len(modality.embedder.embedding_shape) == 1,
            rngs=rngs,
        )
    )
    backbone = DiTBackbone(
        dims,
        modality_num_tokens=modality_num_tokens,
        rngs=rngs,
    )
    context_encoder = SumContextEncoder(
        time_encoder=TimeNoiseContextEncoder(
            dims,
            gamma_fn=gamma_fn,
            rngs=rngs,
        )
    )
    return EncoderBackboneDecoderNetwork(
        encoders=encoders,
        backbone=backbone,
        decoders=decoders,
        context_encoder=context_encoder,
    )

In [ ]:
# Six digit positions are the sequence: one token per position, not squeezed.
mask_network = make_dit_network(
    mask_registry,
    rngs,
    modality_num_tokens=mask_registry.broadcast(SEQ_LEN),
)

### 3.3. The generative model

We use the predefined [`PosteriorMixtureGenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.factory.PosteriorMixtureGenerativeModel). It requires every modality to use a `DiscreteInterpolant` that implements `rates_from_target_posterior`, and it implements both abstract methods:

- **`get_loss`** — softmax cross-entropy of the $K$ posterior logits against the one-hot target. For a mask interpolant the mask channel is dropped from the target, and the loss is applied only on positions that are *still masked* (revealed tokens contribute nothing).
- **`get_generator`** — softmax the logits and call `rates_from_target_posterior` to obtain per-modality `TransitionRates` (forward and backward).

These are linked: the loss trains a denoising posterior, and the generator is that posterior turned into CTMC rates.

In [ ]:
mask_gen_model = PosteriorMixtureGenerativeModel(
    network=mask_network,
    modality_registry=mask_registry,
)

### 3.4. Training

Training is the same [`TrainingLoop`](https://instadeepai.github.io/stix/api_reference/training/training_loop.html#stix.training.training_loop.TrainingLoop) + [`LossPipeline`](https://instadeepai.github.io/stix/api_reference/training/loss_pipeline.html) setup as [tutorial 1](./1.training_and_sampling.ipynb): Adam, an EMA copy of the parameters, and periodic eval. We train for `NUM_STEPS` steps and keep the EMA model for sampling.


In [ ]:
NUM_STEPS = 8000
EVAL_EVERY_N_STEPS = 1000
JOINT_NUM_STEPS = 2000


def make_history_io_handler(verbose=True):
    """A TrainingIOHandler that records a plottable {step, train_loss, eval_loss} history."""
    history, by_step = [], {}

    def _collect(category, to_log, step):
        entry = by_step.setdefault(step, {"step": step})
        if entry not in history:
            history.append(entry)
        if category == LoggingCategory.TRAIN_METRICS:
            entry["train_loss"] = to_log["loss"]
        elif category == LoggingCategory.EVAL_METRICS:
            entry["eval_loss"] = to_log["loss"]

    io_handler = TrainingIOHandler()
    io_handler.attach_logger(_collect)
    if verbose:
        io_handler.attach_logger(log_metrics_to_line)
    return history, io_handler


def train_model(
    gen_model, train_data, val_data, num_steps=None, eval_every_n_steps=None
):
    """Run a TrainingLoop and return (ema_model, history)."""
    num_steps = NUM_STEPS if num_steps is None else num_steps
    eval_every_n_steps = (
        EVAL_EVERY_N_STEPS if eval_every_n_steps is None else eval_every_n_steps
    )
    history, io_handler = make_history_io_handler()
    training_loop = TrainingLoop(
        train_data=train_data,
        val_data=val_data,
        loss_pipeline=LossPipeline(),
        gen_model=gen_model,
        optimizer_tx=optax.adam(1e-3),
        config=TrainingLoopConfig(
            num_steps=num_steps,
            eval_every_n_steps=eval_every_n_steps,
            ema_decay=0.999,
        ),
        io_handler=io_handler,
    )
    training_loop.run()
    return training_loop.ema_model, history


mask_ema_model, mask_history = train_model(mask_gen_model, train_iter, validation_iter)

### 3.5. Sampling

Discrete interpolants yield `TransitionRates`, which can only be handled using the [`ManualSolver`](https://instadeepai.github.io/stix/api_reference/sampling/solver_manual.html#stix.sampling.solver_manual.ManualSolver) that can step a CTMC (the diffrax [`Solver`](https://instadeepai.github.io/stix/api_reference/sampling/solver.html#stix.sampling.solver.Solver) uses adaptive steps to integrate S/ODEs and is not compatible with CTMC).

`ModalityRegistry.sample_initial_state` draws $z_0$ with every position starting in the mask state. We `vmap` the solver over samples, decode the six digits back to $(a,b,c)$, and report the fraction of samples that satisfy $c = (a \times b) \bmod 100$.


In [ ]:
NUM_SAMPLES = 512


def sample_tokens_model(ema_model, registry, key, num_steps=1000, lambda_fn=None):
    """Draw token sequences by vmapping the CTMC solver over samples.

    `lambda_fn` is the per-modality corrector scale; the default (None)
    means lambda = 0, i.e. forward rates only.
    """
    solver = ManualSolver(
        ManualSolverConfig(
            num_steps=num_steps,
            direction=Direction.FORWARD,
            t_max_tolerance=0.01,
            stochasticity_scale=lambda_fn,
        )
    )
    key_init, key_solve = jr.split(key)
    z_init = registry.sample_initial_state(key_init, num_samples=NUM_SAMPLES)
    solve_keys = jr.split(key_solve, NUM_SAMPLES)
    raw_samples = jax.vmap(lambda x, k: solver(ema_model, x, k))(z_init, solve_keys)
    return raw_samples["tokens"]  # (NUM_SAMPLES, SEQ_LEN), integer indices


def decode_abc(tokens):
    """Decode 6-digit sequences into integers (a, b, c)."""
    a = tokens[:, 0] * 10 + tokens[:, 1]
    b = tokens[:, 2] * 10 + tokens[:, 3]
    c = tokens[:, 4] * 10 + tokens[:, 5]
    return a, b, c


def plot_token_stats(tokens, title):
    """Report the fraction of samples satisfying c = (a * b) % 100 and show examples."""
    a, b, c = decode_abc(tokens)
    correct = c == (a * b) % 100
    accuracy = jnp.mean(correct)

    # A few decoded examples (correct ones marked).
    examples = "\n".join(
        f"  {int(ai):2d} x {int(bi):2d} = {int(ai) * int(bi) % 100:2d}  "
        f"(model said {int(ci):2d}) {'ok' if ok else 'x'}"
        for ai, bi, ci, ok in zip(a[:6], b[:6], c[:6], correct[:6])
    )
    print(f"{title}: valid multiplications = {accuracy:.1%}\n{examples}")

    counts = jnp.bincount(tokens.reshape(-1), length=NUM_CATEGORIES)
    freqs = counts / counts.sum()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(jnp.arange(NUM_CATEGORIES), freqs, width=0.6, color="tab:blue")
    ax.axhline(1 / NUM_CATEGORIES, ls="--", c="grey", label="uniform")
    ax.set_title(f"{title}\nvalid multiplications: {accuracy:.1%}")
    ax.set_xlabel("digit")
    ax.set_ylabel("frequency (all positions)")
    ax.set_xticks(list(range(NUM_CATEGORIES)))
    ax.legend()
    plt.tight_layout()
    plt.show()


key, mask_sample_key = jr.split(key)
mask_tokens = sample_tokens_model(mask_ema_model, mask_registry, mask_sample_key)
plot_token_stats(mask_tokens, "Masked diffusion")

### 3.6. Corrector sampling: putting the backward rates to work

The run above used the forward rates only. The solver actually steps the mix

$$\bar u_t = (1 + \lambda_t)\,\hat u_t + \lambda_t\,\check u_t,$$

set per modality by `stochasticity_scale` (defaulting to $\lambda \equiv 0$). Since $\hat u_t + \check u_t$ is divergence-free against $p_t$, every $\lambda_t \geq 0$ leaves the marginals unchanged, which is why the mix needs a single weight: $\lambda_t$ is pure extra stochasticity, and it is the very same knob that scales the SDE noise of a continuous modality. Concretely, for masked diffusion $\lambda_t > 0$ re-masks a fraction of the already-revealed digits and lets the forward term decide them again — a token committed early, before the rest of the sequence pinned it down, gets a second chance.

Below we sample the same trained model with $\lambda = 0.5$, everything else unchanged.

In [ ]:
# Any lambda >= 0 keeps the marginals; it alone sets the stochasticity.
LAMBDA = 0.5

key, corrector_key = jr.split(key)
corrector_tokens = sample_tokens_model(
    mask_ema_model,
    mask_registry,
    corrector_key,
    lambda_fn={"tokens": lambda t: LAMBDA},
)
plot_token_stats(corrector_tokens, f"Masked diffusion, corrector (lambda={LAMBDA})")

## 4. Uniform diffusion

Uniform diffusion uses the same two-point schedule, but mixes the target with a *uniform* source instead of a mask. The state space is exactly the $K$ data categories (no extra symbol). Tokens are again integer indices, **one-hot** encoded with width $K$. The interpolant path of [`UniformDiscreteInterpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.UniformDiscreteInterpolant) is

$$p_t(z \mid z_{\mathrm{tgt}}) = \kappa_t\,\delta_{z_{\mathrm{tgt}}}(z) + (1-\kappa_t)\,\mathrm{Unif}(\{0,\dots,K-1\}).$$

At $t=0$, $z_0$ is drawn uniformly; at $t=1$ it is the data token. The probability velocity has the same closed form as in section 3,

$$\hat{u}_t = \frac{\dot\kappa_t}{1-\kappa_t}\bigl(p_{\mathrm{tgt}\mid t}(\,\cdot\mid z_t) - \delta_{z_t}\bigr),$$

but now jumps are allowed from *any* current state (there is no absorbing mask). The network still predicts a posterior over the $K$ categories — which here coincide with the CTMC states.

### 4.1 Modality registry, network and generative model

We repeat the section 3 recipe, changing one line: the interpolant is now a `UniformDiscreteInterpolant`. Because the embedder factory reads `num_states` off the interpolant, the one-hot width follows automatically — $K$ here rather than $K+1$. The network and generative model are built exactly as before.

In [ ]:
# The same recipe as section 3. The only line that differs is the interpolant:
# a UniformDiscreteInterpolant, whose `num_states` is K rather than K + 1.
uniform_registry = ModalityRegistry.from_batch(batch)
uniform_registry.set(
    "interpolant",
    UniformDiscreteInterpolant(num_categories=NUM_CATEGORIES, kappa_fn=lambda t: t),
)
uniform_registry.set(
    field="embedder",
    value=lambda modality: OneHotDiscreteEmbedder(
        dm_shape=modality.shape,
        num_states=modality.interpolant.num_states,
    ),
    is_factory=True,
)
uniform_registry.set("num_categories", NUM_CATEGORIES)

key, uniform_net_key = jr.split(key)
uniform_network = make_dit_network(
    uniform_registry,
    nnx.Rngs(uniform_net_key),
    modality_num_tokens=uniform_registry.broadcast(SEQ_LEN),
)

uniform_gen_model = PosteriorMixtureGenerativeModel(
    network=uniform_network,
    modality_registry=uniform_registry,
)

### 4.2 Training

Same `train_model` helper as section 3.

In [ ]:
uniform_ema_model, _ = train_model(uniform_gen_model, train_iter, validation_iter)

### 4.3 Sampling

Same CTMC sampling path. The initial state is now a uniform draw over the digits rather than the mask symbol. We plot the same multiplication accuracy and digit frequencies.

In [ ]:
key, uniform_sample_key = jr.split(key)
uniform_tokens = sample_tokens_model(
    uniform_ema_model, uniform_registry, uniform_sample_key
)
plot_token_stats(uniform_tokens, "Uniform diffusion")

## 5. Multi-modal joint model

So far both models were all-discrete. `stix` can mix continuous and discrete modalities in a single [`VelocityAndPosteriorGenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.factory.VelocityAndPosteriorGenerativeModel): continuous leaves behave like [`VelocityOneSidedGenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.factory.VelocityOneSidedGenerativeModel) (velocity MSE, `VelocityAndScore` generator), discrete leaves like `PosteriorMixtureGenerativeModel` (denoising cross-entropy, `TransitionRates` generator). Dispatch is per interpolant.

Joint sampling needs a solver that can step an S/ODE and a CTMC on the *same* time grid. That is exactly [`ManualSolver`](https://instadeepai.github.io/stix/api_reference/sampling/solver_manual.html#stix.sampling.solver_manual.ManualSolver): it picks an Euler / Euler–Maruyama / categorical jump per modality from the generator type. The parameter `stochasticity_scale` is nonsensical for CTMC sampling, so discrete leaves must set it to `None`; continuous leaves take any valid schedule to sample an SDE, or `None` for ODE sampling.

### 5.1 Gaussian mixture toy dataset

We reuse the four-mode GMM of [tutorial 1](./1.training_and_sampling.ipynb): `coordinates` are 2D points around the corners of a square, `index` is the mode they were drawn from. The discrete modality is stored as an **integer index** (shape `(B,)`, not one-hot), matching the index-valued convention of sections 3–4.

In [ ]:
CORNERS = jnp.array(
    [[-1.0, -1.0], [1.0, -1.0], [-1.0, 1.0], [1.0, 1.0]], dtype=jnp.float32
)
NUM_MODES = len(CORNERS)


def sample_coord_and_index(key):
    """Pick a corner, add Gaussian noise; return (coordinates, integer mode index)."""
    idx_key, noise_key = jr.split(key)
    idx = jr.randint(idx_key, (), 0, NUM_MODES)
    coordinates = jr.normal(noise_key, (2,), dtype=jnp.float32) * 0.2 + CORNERS[idx]
    return coordinates, idx.astype(jnp.int32)


@jax.jit
def sample_gmm(batch_indices, key):
    """Build a Batch pairing continuous coordinates with an integer mode index."""
    keys = jax.vmap(partial(jax.random.fold_in, key))(batch_indices)
    coordinates, idx = jax.vmap(sample_coord_and_index)(keys)
    return Batch(
        raw_batch={
            "coordinates": RawSourceTargetPair(target=coordinates, source=None),
            # Discrete modality as an integer index (shape (B,)), not one-hot.
            "index": RawSourceTargetPair(target=idx, source=None),
        },
        is_discrete={"coordinates": False, "index": True},
    )


def gmm_dataset(seed):
    """An infinite, shuffled, batched stream of (coordinates, mode index) pairs."""
    return (
        grain.MapDataset.range(int(1e9))
        .seed(seed)
        .shuffle()
        .repeat()
        .batch(batch_size, drop_remainder=True)
        .map(partial(sample_gmm, key=jr.key(seed)))
        .to_iter_dataset()
    )


gmm_train_iter = iter(gmm_dataset(seed=0))
gmm_val_iter = iter(gmm_dataset(seed=42))

### 5.2 Modality registry, network and generative model

We build the registry by hand so each modality gets its own interpolant:

- `coordinates` — [`FlowMatchingOneSidedInterpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.FlowMatchingOneSidedInterpolant) + [`IdentityEmbedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.IdentityEmbedder).
- `index` — `MaskDiscreteInterpolant` with $K=$ `NUM_MODES` + `OneHotDiscreteEmbedder` of width $K+1$ (data modes + mask).

Each modality is a single token (`squeeze_sequence=True`). The DiT decoder emits a 2D velocity for `coordinates` and $K$ logits for `index`.

In [ ]:
# Same `from_batch` + `.set` machinery as sections 3-4, but each `.set` is now
# filtered, so the continuous and discrete modalities get different values for the
# same field. The interpolants must be set before the embedders: the discrete
# embedder factory reads `num_states` off its modality's interpolant.
def is_discrete(modality):
    """Filter selecting the discrete modalities of the registry."""
    return modality.is_discrete


def is_continuous(modality):
    """Filter selecting the continuous modalities of the registry."""
    return not modality.is_discrete


joint_registry = ModalityRegistry.from_batch(next(gmm_train_iter))
joint_registry.set(
    "interpolant", FlowMatchingOneSidedInterpolant(), filter_fn=is_continuous
)
joint_registry.set(
    "interpolant",
    MaskDiscreteInterpolant(num_categories=NUM_MODES, kappa_fn=lambda t: t),
    filter_fn=is_discrete,
)
# Index-valued discrete data: `from_batch` cannot infer K, so state it.
joint_registry.set("num_categories", NUM_MODES, filter_fn=is_discrete)
joint_registry.set(
    "embedder",
    lambda modality: IdentityEmbedder(dm_shape=modality.shape),
    filter_fn=is_continuous,
    is_factory=True,
)
# One-hot embedding of width NUM_MODES + 1 (data modes + mask symbol).
joint_registry.set(
    "embedder",
    lambda modality: OneHotDiscreteEmbedder(
        dm_shape=modality.shape, num_states=modality.interpolant.num_states
    ),
    filter_fn=is_discrete,
    is_factory=True,
)

# `coordinates` is continuous, so its interpolant carries the noise schedule the
# time encoder needs: gamma(t) = 1 - t here, not the placeholder used above.
coordinates_interpolant = joint_registry.map(lambda m: m.interpolant)["coordinates"]

key, joint_net_key = jr.split(key)
joint_network = make_dit_network(
    joint_registry,
    nnx.Rngs(joint_net_key),
    modality_num_tokens=joint_registry.broadcast(1),
    gamma_fn=coordinates_interpolant.gamma_fn,
)
joint_gen_model = VelocityAndPosteriorGenerativeModel(
    network=joint_network,
    modality_registry=joint_registry,
)

### 5.3 Training

Same loop, fewer steps (`JOINT_NUM_STEPS`): the 2D GMM is easier than modular multiplication.

In [ ]:
joint_ema_model, _ = train_model(
    joint_gen_model,
    gmm_train_iter,
    gmm_val_iter,
    num_steps=JOINT_NUM_STEPS,
    eval_every_n_steps=500,
)

### 5.4. Joint sampling

The registry derives the per-modality `stochasticity_scale` itself: an SDE on the continuous leg (scale proportional to that modality's own $\gamma(t)$, as in [tutorial 1](./1.training_and_sampling.ipynb)) and `None` on the discrete leg, i.e. $\lambda \equiv 0$, the CTMC with no corrector. We scatter generated coordinates, coloured by the sampled mode, over a grey cloud of true samples.


In [ ]:
STOCHASTICITY_SCALE = 1.0


def _stochasticity_scale_per_modality(modality):
    """SDE scale from each continuous leg's own gamma(t); discrete legs get none."""
    if is_discrete(modality):
        # None is lambda = 0: the CTMC steps the forward rates alone, no corrector.
        return None
    return lambda t: modality.interpolant.gamma_fn(t) * STOCHASTICITY_SCALE


joint_solver = ManualSolver(
    ManualSolverConfig(
        num_steps=200,
        direction=Direction.FORWARD,
        stochasticity_scale=joint_registry.map(_stochasticity_scale_per_modality),
    )
)

key, key_init, key_solve = jr.split(key, 3)
z_init = joint_registry.sample_initial_state(key_init, num_samples=NUM_SAMPLES)
solve_keys = jr.split(key_solve, NUM_SAMPLES)
joint_samples = jax.vmap(lambda x, k: joint_solver(joint_ema_model, x, k))(
    z_init, solve_keys
)
sampled_coords = joint_samples["coordinates"]  # (NUM_SAMPLES, 2)
sampled_modes = joint_samples["index"]  # (NUM_SAMPLES,) integer mode

In [ ]:
ref_coords = next(gmm_val_iter).raw_batch["coordinates"].target

cmap = plt.get_cmap("tab10", NUM_MODES)
norm = BoundaryNorm([i - 0.5 for i in range(NUM_MODES + 1)], NUM_MODES)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(ref_coords[:, 0], ref_coords[:, 1], s=12, c="lightgrey", label="true")
points = ax.scatter(
    sampled_coords[:, 0],
    sampled_coords[:, 1],
    s=8,
    c=sampled_modes,
    cmap=cmap,
    norm=norm,
    alpha=0.6,
    label="generated",
)
ax.set_title("Joint sampling: SDE coordinates + CTMC mode index")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
ax.legend(loc="upper right")
fig.colorbar(points, ax=ax, label="sampled mode", ticks=range(NUM_MODES))
plt.tight_layout()
plt.show()

## Recap

- **Masked diffusion.** $K+1$ states; the interpolant mixes the target with a mask symbol. The network predicts a $K$-way posterior; `PosteriorMixtureGenerativeModel` turns it into `TransitionRates`. Sample with `ManualSolver` from an all-mask $z_0$.
- **Corrector sampling.** `TransitionRates` carries both the forward velocity $\hat u_t$ and the time-reversed one $\check u_t$. The solver mixes them as $(1+\lambda_t)\,\hat u_t + \lambda_t\,\check u_t$, set by `stochasticity_scale`; every $\lambda_t \geq 0$ preserves the marginals, and $\lambda_t > 0$ re-masks and re-decides. `Direction.REVERSE` swaps the two weights to run the chain backwards.
- **Uniform diffusion.** Same factory and solver; the source is $\mathrm{Unif}(\{0,\dots,K-1\})$ and the state space has $K$ states.
- **Joint model.** `VelocityAndPosteriorGenerativeModel` mixes a continuous one-sided interpolant with a discrete one. `ManualSolver` steps both on one grid (SDE + CTMC).
